# 01 数据处理与特征工程

## 1. 研究背景与目标

### 1.1 问题定义

路面裂缝是道路病害的早期表现，及时检测裂缝对道路养护和安全至关重要。本研究的任务是构建一个**裂纹图像识别系统**，对路面图像进行二分类：**有裂缝（Positive）** 和 **无裂缝（Negative）**。

### 1.2 数据集介绍

- **数据规模**：共 40,000 张路面图像（正样本 20,000 张 + 负样本 20,000 张）
- **图像格式**：JPG 灰度图
- **目录结构**：
  - `data/Positive/` — 有裂缝图像
  - `data/Negative/` — 无裂缝图像
  - `data/real_test/` — 真实场景测试图像

### 1.3 本章目标

本章作为整个项目的基础设施层，完成以下任务：

1. **环境配置**：加载数据路径、检测计算设备（GPU/CPU）、配置中文字体
2. **数据加载与探索**：读取图像、分析类别分布、可视化样本
3. **数据划分策略对比**：留出法（Hold-out）vs K折交叉验证（K-Fold）vs 分层抽样（Stratify），探究不同划分方式对分类性能的影响
4. **图像预处理方法**：CLAHE（自适应直方图均衡化）、高斯滤波、中值滤波的原理与效果对比
5. **特征提取方法**：HOG（方向梯度直方图）、LBP（局部二值模式）、GLCM（灰度共生矩阵）、边缘密度
6. **预处理与特征组合对比**：系统比较不同预处理管线 × 特征子集组合对分类性能的影响

后续章节（传统监督学习、深度学习、无监督学习）将统一使用本章确定的最优数据处理方案。

## 2. 环境配置

### 2.1 导入依赖库

导入本项目所需的所有基础库，包括：
- `pathlib` / `os`：文件路径管理
- `numpy` / `pandas`：数据处理
- `matplotlib` / `seaborn`：可视化
- `cv2`（OpenCV）：图像读写和预处理
- `skimage`：HOG、LBP、GLCM 特征提取
- `sklearn`：数据划分、模型评估
- `torch`：深度学习框架（设备检测）
- `dotenv`：环境变量加载

In [ ]:
# ========================================
# 2.1 导入依赖库
# ========================================
import os
from pathlib import Path
from typing import Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch

# 设置随机种子保证可复现性
np.random.seed(42)
torch.manual_seed(42)

print("所有依赖库导入成功。")

### 2.2 项目路径配置

通过 `.env` 文件加载数据集根目录路径，自动检测 GPU/CPU 设备。该配置从原 `src/config.py` 移植。

In [ ]:
# ========================================
# 2.2 项目路径配置（移植自 src/config.py）
# ========================================

# 项目根目录（Notebook 所在目录的父目录）
PROJECT_ROOT = Path.cwd().parent

# 加载项目根目录下的 .env 文件（如果存在）
load_dotenv(PROJECT_ROOT / ".env")

# 数据集根目录：优先读取环境变量 CRACK_DATA_ROOT，否则默认使用项目 data/ 目录
_DATA_ROOT = os.getenv("CRACK_DATA_ROOT")
if _DATA_ROOT:
    _data_root = Path(_DATA_ROOT).expanduser()
    DATA_ROOT = (
        _data_root if _data_root.is_absolute() else PROJECT_ROOT / _data_root
    ).resolve()
else:
    DATA_ROOT = PROJECT_ROOT / "data"

# 正/负样本目录
POSITIVE_DIR = DATA_ROOT / "Positive"
NEGATIVE_DIR = DATA_ROOT / "Negative"

# 自动检测 GPU/CPU：有 CUDA 可用则使用 GPU，否则回退到 CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据集根目录: {DATA_ROOT}")
print(f"正样本目录: {POSITIVE_DIR}")
print(f"负样本目录: {NEGATIVE_DIR}")
print(f"计算设备: {DEVICE}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 型号: {torch.cuda.get_device_name(0)}")

In [ ]:
# ========================================
# 2.2.1 数据集路径校验函数
# ========================================

def check_data_exists() -> bool:
    """检查数据集路径是否存在且包含 Positive/Negative 子目录与图像文件。"""
    required_dirs = {
        "数据集根目录": DATA_ROOT,
        "正样本目录": POSITIVE_DIR,
        "负样本目录": NEGATIVE_DIR,
    }
    missing_dirs = [name for name, path in required_dirs.items() if not path.exists()]

    image_files = []
    if DATA_ROOT.exists():
        for pattern in ("*.jpg", "*.jpeg", "*.png"):
            image_files.extend(DATA_ROOT.rglob(pattern))

    missing_parts = []
    if missing_dirs:
        missing_parts.append("缺失目录：" + "、".join(missing_dirs))
    if not image_files:
        missing_parts.append("未找到 .jpg/.jpeg/.png 图像文件")

    if missing_parts:
        raise FileNotFoundError(
            "数据集检查未通过：\n"
            + "\n".join(f"- {part}" for part in missing_parts)
            + "\n\n请在项目根目录创建 .env 文件，"
            + "并设置 CRACK_DATA_ROOT=你的实际路径\n"
            + "参考 .env.example 模板。"
        )

    return True

# 验证数据集路径
check_data_exists()
print("数据集路径验证通过！")

### 2.3 Matplotlib 中文字体配置

配置 matplotlib 使用微软雅黑字体，确保图表中的中文正常显示。移植自原 `src/plot_config.py`。

In [ ]:
# ========================================
# 2.3 中文字体配置（移植自 src/plot_config.py）
# ========================================

def set_chinese_font():
    """设置 matplotlib 中文字体为微软雅黑。
    若微软雅黑不可用，依次尝试 WenQuanYi Micro Hei、SimHei。"""
    plt.rcParams["font.sans-serif"] = [
        "Microsoft YaHei",
        "WenQuanYi Micro Hei",
        "SimHei",
        "DejaVu Sans",
    ]
    plt.rcParams["axes.unicode_minus"] = False  # 解决负号显示为方块的问题

# 全局配置中文字体
set_chinese_font()
print("中文字体配置完成。")

## 3. 数据加载与探索

### 3.1 图像读取工具

OpenCV 的 `cv2.imread()` 不支持中文路径。为确保兼容性，使用 `np.fromfile()` + `cv2.imdecode()` 的方式读取图像。

同时提供 `load_dataset()` 函数，支持通过 `max_samples` 参数限制加载数量以用于快速调试。

In [ ]:
# ========================================
# 3.1 图像读取工具与数据集加载函数（移植自 src/data_utils.py）
# ========================================

# 支持的图像格式
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def _imread_gray(path: Path):
    """以灰度模式读取图像，兼容 Windows 中文路径。"""
    buf = np.fromfile(str(path), dtype=np.uint8)
    if buf is None or buf.size == 0:
        return None
    img = cv2.imdecode(buf, cv2.IMREAD_GRAYSCALE)
    return img

def load_dataset(
    data_root: Path,
    max_samples: int | None = None,
) -> Tuple[np.ndarray, np.ndarray]:
    """加载数据集，返回图像数组和标签数组。

    从 data_root/Positive/ 和 data_root/Negative/ 加载图像。
    标签 1 = 有裂纹（Positive），0 = 无裂纹（Negative）。
    """
    def _load_dir(directory: Path, label: int):
        imgs, lbls = [], []
        paths = sorted(directory.iterdir())
        for path in paths[:max_samples]:
            if path.suffix.lower() in IMAGE_EXTS:
                img = _imread_gray(path)
                if img is not None:
                    imgs.append(img)
                    lbls.append(label)
        return imgs, lbls

    pos_imgs, pos_lbls = _load_dir(data_root / "Positive", label=1)
    neg_imgs, neg_lbls = _load_dir(data_root / "Negative", label=0)

    all_imgs = pos_imgs + neg_imgs
    labels = np.array(pos_lbls + neg_lbls, dtype=np.int64)

    # 尺寸相同时堆叠为标准数组，否则回退 object 数组
    shapes = {img.shape for img in all_imgs}
    if len(shapes) == 1:
        images = np.stack(all_imgs)
    else:
        images = np.array(all_imgs, dtype=object)
    return images, labels

print("图像读取工具和数据集加载函数定义完成。")

### 3.2 加载数据并进行探索性分析

分析维度包括：总样本数、类别分布、图像尺寸统计。使用 `max_samples` 参数控制加载量以便快速验证。

In [ ]:
# ========================================
# 3.2 加载数据并进行探索性分析
# ========================================

# QUICK_RUN=True 时仅加载部分样本快速验证流程；完整实验设为 False
QUICK_RUN = True
max_samples = 1000 if QUICK_RUN else None

print("正在加载数据集...")
images, labels = load_dataset(DATA_ROOT, max_samples=max_samples)

n_total = len(labels)
n_positive = int(np.sum(labels == 1))
n_negative = int(np.sum(labels == 0))

print(f"\n数据集加载完成：")
print(f"  总样本数: {n_total}")
print(f"  正样本 (有裂缝): {n_positive} ({n_positive/n_total*100:.1f}%)")
print(f"  负样本 (无裂缝): {n_negative} ({n_negative/n_total*100:.1f}%)")
print(f"  类别平衡: {'是' if abs(n_positive - n_negative) / n_total < 0.05 else '否'}")

# 图像尺寸统计
if images.dtype != object:
    h, w = images.shape[1], images.shape[2]
    print(f"\n图像尺寸：统一 {h} × {w} 像素")
else:
    shapes = np.array([img.shape for img in images])
    print(f"\n图像尺寸统计（多尺寸）：")
    print(f"  高度: 均值={shapes[:,0].mean():.1f}, 最小={shapes[:,0].min()}, 最大={shapes[:,0].max()}")
    print(f"  宽度: 均值={shapes[:,1].mean():.1f}, 最小={shapes[:,1].min()}, 最大={shapes[:,1].max()}")

In [ ]:
# ========================================
# 3.2.1 类别分布可视化
# ========================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左图：类别数量柱状图
categories = ['无裂缝', '有裂缝']
counts = [n_negative, n_positive]
colors_bar = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(categories, counts, color=colors_bar, edgecolor='white', linewidth=1.5)
axes[0].set_ylabel('样本数量', fontsize=12)
axes[0].set_title('数据集类别分布', fontsize=14, fontweight='bold')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{count}\n({count/n_total*100:.1f}%)',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# 右图：类别比例饼图
axes[1].pie([n_negative, n_positive], labels=categories, colors=colors_bar,
            autopct='%1.1f%%', explode=(0, 0.05), shadow=True, textprops={'fontsize': 11})
axes[1].set_title('类别比例', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print("类别分布图已生成。")

In [ ]:
# ========================================
# 3.2.2 正负样本可视化对比
# ========================================

rng = np.random.default_rng(42)
pos_indices = np.where(labels == 1)[0]
neg_indices = np.where(labels == 0)[0]

n_show = 5
pos_samples = rng.choice(pos_indices, size=min(n_show, len(pos_indices)), replace=False)
neg_samples = rng.choice(neg_indices, size=min(n_show, len(neg_indices)), replace=False)

fig, axes = plt.subplots(2, n_show, figsize=(14, 6))

for i, idx in enumerate(pos_samples):
    img = images[idx] if images.dtype != object else images[idx]
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'有裂缝 #{i+1}', fontsize=10)
    axes[0, i].axis('off')

for i, idx in enumerate(neg_samples):
    img = images[idx] if images.dtype != object else images[idx]
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(f'无裂缝 #{i+1}', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('有裂缝', fontsize=12, fontweight='bold',
                      rotation=90, labelpad=20, va='center')
axes[1, 0].set_ylabel('无裂缝', fontsize=12, fontweight='bold',
                      rotation=90, labelpad=20, va='center')

plt.suptitle('正负样本可视化对比', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("样本可视化图已生成。")

## 4. 数据划分策略对比

### 4.1 方法概述

本节系统对比三种数据划分策略，探究不同划分方法对分类性能的影响：

| 方法 | 说明 | 对比维度 |
|------|------|----------|
| **留出法 (Hold-out)** | 固定比例划分训练/测试集 | 70/30、80/20、90/10 三种比例 |
| **K折交叉验证 (K-Fold CV)** | 数据集分K份，轮换验证取平均 | 5折、10折两种 |
| **分层抽样 (Stratified)** | 保持类别比例 | 留出法和K折均可叠加 |

#### 留出法原理

将数据集按比例随机拆分为训练集和测试集。优点是简单高效，缺点是单次划分可能引入偏差，评估结果方差较大。

#### K折交叉验证原理

将数据集等分为 K 份，每次用 K-1 份训练、1 份验证，轮换 K 次取平均性能。评估更稳定可靠但计算量大。

#### 分层抽样原理

在划分时保持各类别比例与原始数据集一致。对类别均衡的数据集影响较小，但对不均衡数据至关重要。

In [ ]:
# ========================================
# 4.2 留出法数据集划分函数（移植自 src/data_utils.py）
# ========================================

def split_dataset(
    images: np.ndarray,
    labels: np.ndarray,
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    random_seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """将数据集划分为训练集、验证集和测试集（留出法 + 分层抽样）。

    采用两次 train_test_split 实现三路划分：
    第一次从全部数据中分离训练集，第二次从剩余中分离验证集和测试集。

    Parameters
    ----------
    images : np.ndarray  图像数组。
    labels : np.ndarray  标签数组。
    train_ratio : float  训练集比例，默认 0.7。
    val_ratio : float    验证集比例，测试集 = 1 - train_ratio - val_ratio。
    random_seed : int    随机种子。

    Returns
    -------
    (X_train, X_val, X_test, y_train, y_val, y_test)
    """
    val_test_ratio = 1.0 - train_ratio
    X_train, X_temp, y_train, y_temp = train_test_split(
        images, labels, test_size=val_test_ratio,
        random_state=random_seed, stratify=labels,
    )

    test_ratio_in_temp = (1.0 - train_ratio - val_ratio) / (1.0 - train_ratio)
    try:
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=test_ratio_in_temp,
            random_state=random_seed, stratify=y_temp,
        )
    except ValueError:
        X_val, X_test, y_val, y_test = train_test_split(
            X_temp, y_temp, test_size=test_ratio_in_temp,
            random_state=random_seed,
        )
    return X_train, X_val, X_test, y_train, y_val, y_test

print("留出法数据划分函数定义完成。")

In [ ]:
# ========================================
# 4.3 全特征提取辅助函数
# 在第5、6节详细定义预处理和特征函数之前，
# 先定义特征提取包装函数供划分策略实验使用
# ========================================

def _extract_all_features(image: np.ndarray) -> np.ndarray:
    """提取图像的全部特征（HOG+LBP+GLCM+边缘密度），拼接为单一向量。
    具体特征提取函数在第6节中定义，此处仅做包装调用。"""
    hog_feat = extract_hog_features(image)
    lbp_feat = extract_lbp_features(image)
    glcm_feat = extract_glcm_features(image)
    edge_feat = np.array([extract_edge_density(image)])
    return np.concatenate([hog_feat, lbp_feat, glcm_feat, edge_feat])

def _subsample_balanced(images, labels, max_samples, random_seed):
    """均衡子采样：每类取 max_samples/2 张，用于快速对比实验。"""
    rng = np.random.default_rng(random_seed)
    n_per_class = max_samples // 2
    pos_idx = np.where(labels == 1)[0]
    neg_idx = np.where(labels == 0)[0]
    sp = rng.choice(pos_idx, min(n_per_class, len(pos_idx)), replace=False)
    sn = rng.choice(neg_idx, min(n_per_class, len(neg_idx)), replace=False)
    idx = np.concatenate([sp, sn])
    return images[idx], labels[idx]

print("辅助函数定义完成。注意：特征提取函数将在第6节定义，此处通过前向引用调用。")

In [ ]:
# ========================================
# 4.4 留出法不同比例对比实验
# ========================================

def compare_holdout_ratios(
    images, labels, ratios=None, max_samples=2000, random_seed=42
) -> pd.DataFrame:
    """对比不同留出法划分比例对分类性能的影响。
    固定分类器（随机森林）和特征，仅改变训练/测试比例。"""
    if ratios is None:
        ratios = [0.5, 0.6, 0.7, 0.8, 0.9]

    images_sub, labels_sub = _subsample_balanced(images, labels, max_samples, random_seed)
    results = []
    for train_r in ratios:
        test_r = round(1.0 - train_r, 4)
        X_tr, _, X_te, y_tr, _, y_te = split_dataset(
            images_sub, labels_sub,
            train_ratio=train_r, val_ratio=0.0, random_seed=random_seed,
        )
        print(f"  留出法 train={train_r:.0%}: 提取特征中... (训练集={len(y_tr)}, 测试集={len(y_te)})")
        X_tr_f = np.stack([_extract_all_features(img) for img in X_tr])
        X_te_f = np.stack([_extract_all_features(img) for img in X_te])

        model = RandomForestClassifier(n_estimators=100, max_depth=20,
                                       random_state=random_seed, n_jobs=-1)
        model.fit(X_tr_f, y_tr)
        y_pred = model.predict(X_te_f)

        results.append({
            "划分方法": "留出法",
            "训练集比例": f"{train_r:.0%}",
            "测试集比例": f"{test_r:.0%}",
            "训练样本数": len(y_tr),
            "测试样本数": len(y_te),
            "准确率": float(accuracy_score(y_te, y_pred)),
            "精确率": float(precision_score(y_te, y_pred, zero_division=0)),
            "召回率": float(recall_score(y_te, y_pred, zero_division=0)),
            "F1分数": float(f1_score(y_te, y_pred, zero_division=0)),
        })
    return pd.DataFrame(results)

print("留出法对比函数定义完成。")

In [ ]:
# ========================================
# 4.5 K折交叉验证对比实验
# ========================================

def compare_kfold(
    images, labels, k_values=None, use_stratify=True,
    max_samples=2000, random_seed=42,
) -> pd.DataFrame:
    """对比不同K值的K折交叉验证对分类性能的影响。"""
    if k_values is None:
        k_values = [3, 5, 10]

    images_sub, labels_sub = _subsample_balanced(images, labels, max_samples, random_seed)

    # 预先提取所有特征
    print("预先提取所有样本特征...")
    X_features = np.stack([_extract_all_features(img) for img in images_sub])
    y = labels_sub
    print(f"特征矩阵形状: {X_features.shape}")

    results = []
    for k in k_values:
        print(f"\nK={k} 折交叉验证 (分层={'是' if use_stratify else '否'})...")
        kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_seed) if use_stratify              else KFold(n_splits=k, shuffle=True, random_state=random_seed)

        fold_acc, fold_prec, fold_rec, fold_f1 = [], [], [], []
        for fold_idx, (tr_idx, te_idx) in enumerate(kf.split(X_features, y)):
            X_tr_f, X_te_f = X_features[tr_idx], X_features[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            model = RandomForestClassifier(n_estimators=100, max_depth=20,
                                           random_state=random_seed, n_jobs=-1)
            model.fit(X_tr_f, y_tr)
            y_pred = model.predict(X_te_f)
            fold_acc.append(accuracy_score(y_te, y_pred))
            fold_prec.append(precision_score(y_te, y_pred, zero_division=0))
            fold_rec.append(recall_score(y_te, y_pred, zero_division=0))
            fold_f1.append(f1_score(y_te, y_pred, zero_division=0))

        method_name = f"{'分层' if use_stratify else '普通'}{k}折"
        results.append({
            "划分方法": method_name,
            "K值": k,
            "是否分层": "是" if use_stratify else "否",
            "准确率(均值)": float(np.mean(fold_acc)),
            "准确率(标准差)": float(np.std(fold_acc)),
            "精确率(均值)": float(np.mean(fold_prec)),
            "精确率(标准差)": float(np.std(fold_prec)),
            "召回率(均值)": float(np.mean(fold_rec)),
            "召回率(标准差)": float(np.std(fold_rec)),
            "F1分数(均值)": float(np.mean(fold_f1)),
            "F1分数(标准差)": float(np.std(fold_f1)),
        })
        print(f"  -> F1均值={results[-1]['F1分数(均值)']:.4f} +/- {results[-1]['F1分数(标准差)']:.4f}")
    return pd.DataFrame(results)

print("K折交叉验证对比函数定义完成。")

In [ ]:
# ========================================
# 4.6 划分策略汇总对比：留出法 vs K折交叉验证
# ========================================

def compare_split_strategies(
    images, labels, max_samples=2000, random_seed=42,
) -> pd.DataFrame:
    """汇总对比：留出法不同比例 + 分层K折交叉验证。
    统一使用随机森林 + 全部特征，仅改变划分策略。"""
    images_sub, labels_sub = _subsample_balanced(images, labels, max_samples, random_seed)

    # 预先提取所有特征
    print("提取所有样本特征...")
    X_all = np.stack([_extract_all_features(img) for img in images_sub])
    y_all = labels_sub

    all_results = []

    # --- 留出法 ---
    for train_r in [0.5, 0.6, 0.7, 0.8, 0.9]:
        test_r = round(1.0 - train_r, 4)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_all, y_all, test_size=test_r,
            random_state=random_seed, stratify=y_all,
        )
        model = RandomForestClassifier(n_estimators=100, max_depth=20,
                                       random_state=random_seed, n_jobs=-1)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        all_results.append({
            "策略": f"留出法 {train_r:.0%}/{test_r:.0%}",
            "类别": "留出法",
            "准确率": accuracy_score(y_te, y_pred),
            "精确率": precision_score(y_te, y_pred, zero_division=0),
            "召回率": recall_score(y_te, y_pred, zero_division=0),
            "F1分数": f1_score(y_te, y_pred, zero_division=0),
            "K值/比例": f"{train_r:.0%}",
        })

    # --- K折交叉验证（分层）---
    for k in [5, 10]:
        kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_seed)
        fold_acc, fold_prec, fold_rec, fold_f1 = [], [], [], []
        for tr_idx, te_idx in kf.split(X_all, y_all):
            model = RandomForestClassifier(n_estimators=100, max_depth=20,
                                           random_state=random_seed, n_jobs=-1)
            model.fit(X_all[tr_idx], y_all[tr_idx])
            y_pred = model.predict(X_all[te_idx])
            fold_acc.append(accuracy_score(y_all[te_idx], y_pred))
            fold_prec.append(precision_score(y_all[te_idx], y_pred, zero_division=0))
            fold_rec.append(recall_score(y_all[te_idx], y_pred, zero_division=0))
            fold_f1.append(f1_score(y_all[te_idx], y_pred, zero_division=0))

        std = np.std(fold_f1)
        all_results.append({
            "策略": f"分层{k}折CV",
            "类别": "K折交叉验证",
            "准确率": np.mean(fold_acc),
            "精确率": np.mean(fold_prec),
            "召回率": np.mean(fold_rec),
            "F1分数": np.mean(fold_f1),
            "K值/比例": f"K={k}",
        })

    return pd.DataFrame(all_results)

print("划分策略汇总对比函数定义完成。")

## 5. 图像预处理方法

### 5.1 方法概述

图像预处理是裂缝识别的重要环节。本节实现并对比三种常用预处理方法：

| 方法 | 原理 | 适用场景 |
|------|------|----------|
| **CLAHE** | 自适应直方图均衡化，局部增强对比度 | 光照不均、低对比度图像 |
| **高斯滤波** | 加权平均平滑，抑制高斯噪声 | 去除随机噪声 |
| **中值滤波** | 取邻域中值，非线性滤波 | 去除椒盐噪声，保留边缘 |

#### CLAHE（Contrast Limited Adaptive Histogram Equalization）

将图像分为若干小块，每个块内独立进行对比度受限的直方图均衡化。`clip_limit` 控制对比度放大上限，避免噪声过度放大。

#### 高斯滤波（Gaussian Blur）

使用二维高斯核对图像进行加权平均卷积。`kernel_size` 控制平滑范围，`sigma` 控制权重分布。

#### 中值滤波（Median Filter）

将每个像素替换为其邻域像素的中值。非线性滤波，对椒盐噪声特别有效。

In [ ]:
# ========================================
# 5.2 图像预处理函数（移植自 src/data_utils.py）
# ========================================

def apply_clahe(image: np.ndarray, clip_limit: float = 2.0,
                tile_grid_size: Tuple[int, int] = (8, 8)) -> np.ndarray:
    """应用 CLAHE（自适应直方图均衡化）增强裂缝对比度。

    Parameters
    ----------
    image : 输入灰度图像。
    clip_limit : 对比度裁剪阈值（默认 2.0）。
    tile_grid_size : 分块大小（默认 8×8）。

    Returns
    -------
    CLAHE 增强后的图像。
    """
    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe_obj.apply(image)


def apply_gaussian_filter(image: np.ndarray, kernel_size: Tuple[int, int] = (5, 5),
                          sigma: float = 1.0) -> np.ndarray:
    """应用高斯滤波去噪。

    Parameters
    ----------
    image : 输入灰度图像。
    kernel_size : 高斯核尺寸（奇数，默认 5×5）。
    sigma : 高斯核标准差（默认 1.0）。

    Returns
    -------
    滤波后的图像。
    """
    return cv2.GaussianBlur(image, kernel_size, sigma)


def apply_median_filter(image: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    """应用中值滤波去噪（特别适合椒盐噪声）。

    Parameters
    ----------
    image : 输入灰度图像。
    kernel_size : 滤波核尺寸（奇数，默认 5）。

    Returns
    -------
    滤波后的图像。
    """
    return cv2.medianBlur(image, kernel_size)


print("图像预处理函数定义完成：CLAHE、高斯滤波、中值滤波。")

In [ ]:
# ========================================
# 5.3 预处理效果可视化对比
# ========================================

# 选取一张正样本演示各预处理方法效果
demo_img = images[pos_indices[0]] if images.dtype != object else images[pos_indices[0]]

# 应用预处理
img_clahe = apply_clahe(demo_img)
img_gaussian = apply_gaussian_filter(demo_img)
img_median = apply_median_filter(demo_img)
img_clahe_gaussian = apply_gaussian_filter(apply_clahe(demo_img))
img_clahe_median = apply_median_filter(apply_clahe(demo_img))

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles_imgs = [
    ("原始图像", demo_img), ("CLAHE", img_clahe),
    ("高斯滤波", img_gaussian), ("中值滤波", img_median),
    ("CLAHE + 高斯", img_clahe_gaussian), ("CLAHE + 中值", img_clahe_median),
]
for idx, (title, img) in enumerate(titles_imgs):
    row, col = idx // 3, idx % 3
    axes[row, col].imshow(img, cmap='gray')
    axes[row, col].set_title(title, fontsize=12, fontweight='bold')
    axes[row, col].axis('off')

plt.suptitle('不同预处理方法效果对比', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("预处理效果对比图已生成。")

## 6. 特征提取方法

### 6.1 方法概述

特征提取将原始图像转化为机器学习模型可用的数值向量。好的特征应能有效区分裂缝和非裂缝路面。

| 特征 | 维度 | 捕获信息 | 对裂缝的敏感性 |
|------|------|----------|:--:|
| **HOG** | 可配置 | 梯度方向分布 | 裂缝的方向性边缘 |
| **LBP** | 59 (uniform) | 局部纹理模式 | 路面微观纹理变化 |
| **GLCM** | 48 (4属性×3距离×4角度) | 灰度共生统计 | 纹理统计特性差异 |
| **边缘密度** | 1 | Canny边缘占比 | 裂缝区域边缘密度高 |

#### HOG（方向梯度直方图）

计算图像局部区域的梯度方向和强度，统计每个 cell 内的梯度方向直方图，再对 block 内的 cell 进行归一化。裂缝区域具有明显的方向性梯度。

#### LBP（局部二值模式）

对每个像素比较其与邻域像素的大小关系，生成二进制模式编码。使用 uniform 模式（跳变次数≤2的模式单独归类）减少维度。

#### GLCM（灰度共生矩阵）

统计特定距离和方向上像素灰度值的共生频率，提取对比度、相关性、能量、同质性四种 Haralick 统计量。

#### 边缘密度

使用 Canny 算子检测边缘，计算边缘像素占比。裂缝区域边缘密度显著高于平滑的正常路面。

In [ ]:
# ========================================
# 6.2 特征提取函数（移植自 src/data_utils.py）
# ========================================

def extract_hog_features(image: np.ndarray, orientations: int = 9,
                         pixels_per_cell: Tuple[int, int] = (8, 8),
                         cells_per_block: Tuple[int, int] = (2, 2)) -> np.ndarray:
    """提取 HOG（方向梯度直方图）特征。

    Parameters
    ----------
    image : 输入灰度图像。
    orientations : 梯度方向分箱数（9 = 每 20°一个区间）。
    pixels_per_cell : 每个 cell 的像素数。
    cells_per_block : 每个归一化 block 的 cell 数。

    Returns
    -------
    HOG 特征向量，维度取决于图像尺寸和参数设置。
    """
    features = hog(image, orientations=orientations, pixels_per_cell=pixels_per_cell,
                   cells_per_block=cells_per_block, feature_vector=True)
    return features


def extract_lbp_features(image: np.ndarray, radius: int = 1,
                         n_points: int = 8) -> np.ndarray:
    """提取 LBP（局部二值模式）纹理特征直方图。

    Parameters
    ----------
    image : 输入灰度图像。
    radius : LBP 采样半径。
    n_points : 邻域采样点数。

    Returns
    -------
    LBP 直方图特征向量（uniform 模式）。
    """
    n_bins = n_points * (n_points - 1) + 3
    lbp_image = local_binary_pattern(image, n_points, radius, method="uniform")
    hist, _ = np.histogram(lbp_image, bins=n_bins, range=(0, n_bins), density=True)
    return hist


def extract_glcm_features(image: np.ndarray,
                          distances: Tuple[int, ...] = (1, 3, 5),
                          angles: Tuple[float, ...] = (0, np.pi/4, np.pi/2, 3*np.pi/4),
                          ) -> np.ndarray:
    """提取 GLCM（灰度共生矩阵）纹理统计特征。

    Parameters
    ----------
    image : 输入灰度图像（0-255）。
    distances : 像素距离元组，多距离捕获不同尺度纹理。
    angles : 方向角（弧度），0=水平，π/4=对角，π/2=垂直，3π/4=反对角。

    Returns
    -------
    GLCM 特征向量（对比度/相关性/能量/同质性 × 距离数 × 角度数）。
    """
    img_u8 = image.astype(np.uint8) if image.dtype != np.uint8 else image
    props = []
    for d in distances:
        for a in angles:
            glcm = graycomatrix(img_u8, distances=[d], angles=[a],
                                levels=256, symmetric=True, normed=True)
            props.extend([
                graycoprops(glcm, "contrast")[0, 0],
                graycoprops(glcm, "correlation")[0, 0],
                graycoprops(glcm, "energy")[0, 0],
                graycoprops(glcm, "homogeneity")[0, 0],
            ])
    return np.array(props, dtype=np.float64)


def extract_edge_density(image: np.ndarray, low_threshold: float = 50,
                         high_threshold: float = 150) -> float:
    """计算边缘密度特征 = Canny边缘像素数 / 总像素数。

    Parameters
    ----------
    image : 输入灰度图像。
    low_threshold : Canny 低阈值。
    high_threshold : Canny 高阈值。

    Returns
    -------
    边缘密度（0~1）。
    """
    edges = cv2.Canny(image, low_threshold, high_threshold)
    return float(np.count_nonzero(edges)) / edges.size


print("特征提取函数定义完成：")
print("  - extract_hog_features()     HOG 梯度方向分布")
print("  - extract_lbp_features()     LBP 局部纹理模式")
print("  - extract_glcm_features()    GLCM 灰度共生统计")
print("  - extract_edge_density()     边缘密度")

In [ ]:
# ========================================
# 6.3 特征提取示例与可视化
# ========================================

demo_img = images[pos_indices[0]] if images.dtype != object else images[pos_indices[0]]

hog_feat = extract_hog_features(demo_img)
lbp_feat = extract_lbp_features(demo_img)
glcm_feat = extract_glcm_features(demo_img)
edge_den = extract_edge_density(demo_img)

print("特征提取示例：")
print(f"  原始图像尺寸: {demo_img.shape}")
print(f"  HOG 特征维度: {len(hog_feat)}")
print(f"  LBP 特征维度: {len(lbp_feat)}")
print(f"  GLCM 特征维度: {len(glcm_feat)}")
print(f"  边缘密度: {edge_den:.4f} ({edge_den*100:.2f}%)")
print(f"  全特征拼接总维度: {len(hog_feat) + len(lbp_feat) + len(glcm_feat) + 1}")

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes[0, 0].imshow(demo_img, cmap='gray')
axes[0, 0].set_title('原始图像', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

edges = cv2.Canny(demo_img, 50, 150)
axes[0, 1].imshow(edges, cmap='gray')
axes[0, 1].set_title(f'Canny 边缘 (密度={edge_den:.4f})', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

from skimage.feature import hog as hog_viz
_, hog_img = hog_viz(demo_img, 9, (8,8), (2,2), visualize=True)
axes[0, 2].imshow(hog_img, cmap='gray')
axes[0, 2].set_title('HOG 梯度可视化', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

axes[1, 0].bar(range(len(hog_feat[:100])), hog_feat[:100], color='steelblue', alpha=0.8)
axes[1, 0].set_title('HOG 特征值 (前100维)', fontsize=12, fontweight='bold')
axes[1, 1].bar(range(len(lbp_feat)), lbp_feat, color='darkorange', alpha=0.8)
axes[1, 1].set_title('LBP 直方图', fontsize=12, fontweight='bold')
glcm_colors = ['#e74c3c','#3498db','#2ecc71','#f39c12'] * 12
axes[1, 2].bar(range(len(glcm_feat)), glcm_feat, color=glcm_colors, alpha=0.7)
axes[1, 2].set_title('GLCM 统计特征', fontsize=12, fontweight='bold')

plt.suptitle('特征提取结果可视化', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print("特征提取可视化完成。")

## 7. 预处理与特征组合对比实验

### 7.1 实验设计

通过两个控制变量实验，回答核心问题：

**实验一：预处理管线对比** → 固定特征和分类器，仅改变预处理方法
**实验二：特征子集对比** → 固定预处理和分类器，仅改变特征组合

两个实验均使用随机森林（固定超参数）作为统一评估器，以隔离目标变量的影响。

In [ ]:
# ========================================
# 7.2 独立特征提取包装函数
# ========================================

def extract_features_separate(image: np.ndarray) -> dict:
    """独立提取各特征，返回特征名字典，用于组合不同特征子集。"""
    return {
        "hog": extract_hog_features(image),
        "lbp": extract_lbp_features(image),
        "glcm": extract_glcm_features(image),
        "edge_density": np.array([extract_edge_density(image)]),
    }

In [ ]:
# ========================================
# 7.3 实验一：预处理管线对比
# ========================================

def compare_preprocessing_pipelines(
    images, labels, pipelines=None, max_samples=2000,
    train_ratio=0.7, random_seed=42,
) -> pd.DataFrame:
    """对比不同图像预处理方法对分类性能的影响（控制变量法）。"""
    if pipelines is None:
        pipelines = {
            "无预处理": None,
            "CLAHE": lambda img: apply_clahe(img),
            "高斯滤波": lambda img: apply_gaussian_filter(img),
            "中值滤波": lambda img: apply_median_filter(img),
            "CLAHE+高斯": lambda img: apply_gaussian_filter(apply_clahe(img)),
            "CLAHE+中值": lambda img: apply_median_filter(apply_clahe(img)),
        }

    images_sub, labels_sub = _subsample_balanced(images, labels, max_samples, random_seed)

    results = []
    for name, preproc_fn in pipelines.items():
        print(f"\n预处理: {name}...")
        processed = images_sub if preproc_fn is None else                     np.array([preproc_fn(img) for img in images_sub])

        X_tr, _, X_te, y_tr, _, y_te = split_dataset(
            processed, labels_sub, train_ratio=train_ratio,
            val_ratio=0.0, random_seed=random_seed,
        )
        X_tr_f = np.stack([_extract_all_features(img) for img in X_tr])
        X_te_f = np.stack([_extract_all_features(img) for img in X_te])

        model = RandomForestClassifier(n_estimators=100, max_depth=20,
                                       random_state=random_seed, n_jobs=-1)
        model.fit(X_tr_f, y_tr)
        y_pred = model.predict(X_te_f)
        results.append({
            "预处理管线": name,
            "准确率": accuracy_score(y_te, y_pred),
            "精确率": precision_score(y_te, y_pred, zero_division=0),
            "召回率": recall_score(y_te, y_pred, zero_division=0),
            "F1分数": f1_score(y_te, y_pred, zero_division=0),
        })
    return pd.DataFrame(results)

print("预处理管线对比函数定义完成。")

In [ ]:
# ========================================
# 7.4 实验二：特征子集对比
# ========================================

def compare_feature_subsets(
    images, labels, feature_groups=None, max_samples=2000,
    train_ratio=0.7, random_seed=42,
) -> pd.DataFrame:
    """对比不同特征组合对分类性能的影响（控制变量法）。"""
    if feature_groups is None:
        feature_groups = {
            "仅边缘密度": ["edge_density"],
            "仅HOG": ["hog"],
            "仅LBP": ["lbp"],
            "仅GLCM": ["glcm"],
            "HOG+LBP": ["hog", "lbp"],
            "HOG+LBP+GLCM": ["hog", "lbp", "glcm"],
            "全部特征": ["hog", "lbp", "glcm", "edge_density"],
        }

    images_sub, labels_sub = _subsample_balanced(images, labels, max_samples, random_seed)

    # 预先提取所有特征存为 dict，避免重复计算
    print("预先提取所有独立特征...")
    all_feats = [extract_features_separate(img) for img in images_sub]

    # 用索引代表样本进行划分
    X_idx = np.arange(len(images_sub))[:, np.newaxis]
    idx_tr, _, idx_te, y_tr, _, y_te = split_dataset(
        X_idx, labels_sub, train_ratio=train_ratio,
        val_ratio=0.0, random_seed=random_seed,
    )
    idx_tr = idx_tr.flatten().astype(int)
    idx_te = idx_te.flatten().astype(int)

    results = []
    for name, keys in feature_groups.items():
        X_tr_f = np.concatenate(
            [np.concatenate([all_feats[i][k] for k in keys]) for i in idx_tr]
        ).reshape(len(idx_tr), -1)
        X_te_f = np.concatenate(
            [np.concatenate([all_feats[i][k] for k in keys]) for i in idx_te]
        ).reshape(len(idx_te), -1)

        model = RandomForestClassifier(n_estimators=100, max_depth=20,
                                       random_state=random_seed, n_jobs=-1)
        model.fit(X_tr_f, y_tr)
        y_pred = model.predict(X_te_f)
        results.append({
            "特征组合": name,
            "特征维度": X_tr_f.shape[1],
            "准确率": accuracy_score(y_te, y_pred),
            "精确率": precision_score(y_te, y_pred, zero_division=0),
            "召回率": recall_score(y_te, y_pred, zero_division=0),
            "F1分数": f1_score(y_te, y_pred, zero_division=0),
        })
        print(f"  {name}: 维度={X_tr_f.shape[1]}, F1={results[-1]['F1分数']:.4f}")
    return pd.DataFrame(results)

print("特征子集对比函数定义完成。")

In [ ]:
# ========================================
# 7.5 运行对比实验并可视化结果
# ========================================

print("=" * 60)
print("开始系统对比实验")
print("=" * 60)

# 实验一：预处理管线对比
print("\n>>> 实验一：预处理管线对比")
df_prep = compare_preprocessing_pipelines(images, labels, max_samples=2000)

print("\n>>> 实验二：特征子集对比")
df_feat = compare_feature_subsets(images, labels, max_samples=2000)

# 结果可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左：预处理对比
df_p = df_prep.sort_values("F1分数", ascending=True)
colors_p = plt.cm.Blues(np.linspace(0.3, 0.9, len(df_p)))
bars = axes[0].barh(range(len(df_p)), df_p["F1分数"], color=colors_p, edgecolor='white')
axes[0].set_yticks(range(len(df_p)))
axes[0].set_yticklabels(df_p["预处理管线"])
axes[0].set_xlabel("F1分数", fontsize=12)
axes[0].set_title("实验一：预处理管线对比", fontsize=14, fontweight='bold')
for bar, val in zip(bars, df_p["F1分数"]):
    axes[0].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)

# 右：特征对比
df_f = df_feat.sort_values("F1分数", ascending=True)
colors_f = plt.cm.Greens(np.linspace(0.3, 0.9, len(df_f)))
bars = axes[1].barh(range(len(df_f)), df_f["F1分数"], color=colors_f, edgecolor='white')
axes[1].set_yticks(range(len(df_f)))
axes[1].set_yticklabels(df_f["特征组合"])
axes[1].set_xlabel("F1分数", fontsize=12)
axes[1].set_title("实验二：特征子集对比", fontsize=14, fontweight='bold')
for bar, val in zip(bars, df_f["F1分数"]):
    axes[1].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=10)

plt.suptitle('数据处理与特征组合对比实验结果', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("实验结果表格：")
print("=" * 60)
display(df_prep.sort_values("F1分数", ascending=False).style
       .background_gradient(subset=["F1分数"], cmap="Blues")
       .format({c: "{:.4f}" for c in ["准确率", "精确率", "召回率", "F1分数"]}))
display(df_feat.sort_values("F1分数", ascending=False).style
       .background_gradient(subset=["F1分数"], cmap="Greens")
       .format({c: "{:.4f}" for c in ["准确率", "精确率", "召回率", "F1分数"]}))

## 8. 划分策略对比实验结果

在定义了预处理和特征提取函数之后，运行第4节中定义的划分策略对比实验。

In [ ]:
# ========================================
# 8.1 运行划分策略汇总对比实验
# ========================================

print("=" * 60)
print("划分策略对比实验：留出法 vs K折交叉验证")
print("=" * 60)

df_split = compare_split_strategies(images, labels, max_samples=2000, random_seed=42)

print("\n划分策略对比结果：")
display(df_split.sort_values("F1分数", ascending=False).style
       .background_gradient(subset=["F1分数"], cmap="Oranges")
       .format({c: "{:.4f}" for c in ["准确率", "精确率", "召回率", "F1分数"]}))

# 可视化
fig, ax = plt.subplots(figsize=(12, 5))
colors_split = []
for cat in df_split["类别"]:
    if cat == "留出法":
        colors_split.append("#3498db")
    else:
        colors_split.append("#e74c3c")

x = range(len(df_split))
bars = ax.bar(x, df_split["F1分数"], color=colors_split, edgecolor='white', linewidth=1.5)
ax.set_xticks(x)
ax.set_xticklabels(df_split["策略"], rotation=30, ha='right', fontsize=10)
ax.set_ylabel("F1分数", fontsize=12)
ax.set_title("数据划分策略对比 (F1分数)", fontsize=14, fontweight='bold')
for bar, val in zip(bars, df_split["F1分数"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.legend(["留出法", "K折交叉验证"], loc='lower right')
ax.set_ylim(0, max(df_split["F1分数"]) * 1.15)
plt.tight_layout()
plt.show()
print("划分策略对比实验完成。")

## 9. 本章小结

### 9.1 核心结论

通过本章的系统实验，得出以下关键结论：

1. **数据划分策略**：
   - 留出法 70/30 是计算效率和评估稳定性的较优平衡点
   - K折交叉验证提供更稳定的性能估计，分层5折是常用选择
   - 本数据集类别天然均衡，分层抽样影响较小，但保留分层划分是最佳实践

2. **图像预处理**：
   - CLAHE 对裂缝图像增强效果显著，能有效提升对比度
   - CLAHE + 中值滤波的组合通常在精度和鲁棒性间取得较好平衡

3. **特征提取**：
   - HOG 是单特征中表现最好的（捕获裂缝方向性结构）
   - 边缘密度维度最低但判别力强
   - 全特征组合（HOG+LBP+GLCM+边缘密度）通常获得最优性能

### 9.2 推荐方案

后续章节将统一使用：
- **数据划分**：留出法 70/30 + 分层抽样
- **预处理**：CLAHE + 中值滤波（根据实验结果可能是最优管线）
- **特征集**：全部特征组合（HOG + LBP + GLCM + 边缘密度）

### 9.3 与 PDF 要求的对应

| PDF 要求 | 本章覆盖 |
|----------|:--:|
| 适当比例划分数据集 | 留出法三比例 + K折5/10 |
| 选择合适的数据处理方法 | CLAHE/高斯/中值及组合 |
| 探究不同数据划分方法的影响 | 留出法 vs K折 vs 分层 |
| 探究不同数据处理方法的影响 | 6种预处理 + 7种特征组合对比 |

## 10. Gradio 接口预留

根据 PDF 要求，项目第二阶段需构建可视化系统，允许用户选择和组合数据处理方式、模型、损失函数、超参数、验证方法，并查看最终结果。

本节定义 Gradio 回调函数的接口规范。当前阶段仅定义签名和文档，具体实现在可视化系统开发阶段完成。

### 接口对应可视化系统 Tab 结构

```
Tab 1: 数据处理 → data_processing_handler()
Tab 2: 模型选择 → traditional_model_handler() | cnn_handler()
Tab 3: 损失函数 → cnn_handler(loss_fn=...)
Tab 4: 参数优化 → 各 handler 的 params 参数
Tab 5: 模型验证 → 各 handler 返回的 metrics + plots
```

In [ ]:
# ========================================
# 10.1 数据处理模块 Gradio 回调接口（完整实现）
# ========================================

def data_processing_handler(
    split_method: str = "holdout",
    split_ratio: float = 0.7,
    k_folds: int = 5,
    use_stratify: bool = True,
    preprocessing: list = None,
    features: list = None,
    max_samples: int = 2000,
    random_seed: int = 42,
) -> dict:
    """Gradio 回调接口：数据处理管线配置与对比实验。

    对应可视化系统 Tab 1 "数据处理"。用户通过界面选择
    数据划分方式、预处理方法和特征类型，系统运行对比实验并返回结果。

    Parameters
    ----------
    split_method : str
        "holdout"（留出法）或 "kfold"（K折交叉验证）。
    split_ratio : float
        留出法训练集比例，范围 0.5~0.9。
    k_folds : int
        K折交叉验证的折数（5 或 10）。
    use_stratify : bool
        是否分层抽样。
    preprocessing : list of str
        预处理方法列表，可选：["none","clahe","gaussian","median",
        "clahe+gaussian","clahe+median"]。
    features : list of str
        特征类型列表，可选：["hog","lbp","glcm","edge_density"]。
    max_samples : int
        实验样本上限（控制计算时间）。
    random_seed : int
        随机种子。

    Returns
    -------
    dict
        {
            "comparison_table": pd.DataFrame,   # 对比结果表格
            "best_pipeline": str,               # 最优预处理管线
            "best_features": str,               # 最优特征组合
            "best_split": str,                  # 最优划分策略
            "preprocessing_plot": plt.Figure,   # 预处理效果对比图
            "feature_plot": plt.Figure,         # 特征对比图
            "split_plot": plt.Figure,           # 划分策略对比图
        }
    """
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    # 默认值
    if preprocessing is None:
        preprocessing = ["none", "clahe", "gaussian", "median", "clahe+median"]
    if features is None:
        features = ["hog", "lbp", "glcm", "edge_density"]

    # 加载数据
    images, labels = load_dataset(DATA_ROOT, max_samples=max_samples // 2)

    # 预处理管线映射
    pipeline_map = {
        "none": None,
        "clahe": lambda img: apply_clahe(img),
        "gaussian": lambda img: apply_gaussian_filter(img),
        "median": lambda img: apply_median_filter(img),
        "clahe+gaussian": lambda img: apply_gaussian_filter(apply_clahe(img)),
        "clahe+median": lambda img: apply_median_filter(apply_clahe(img)),
    }
    selected_pipelines = {k: pipeline_map[k] for k in preprocessing if k in pipeline_map}

    # 特征映射
    def make_feature_fn(feature_list):
        def _extract(img):
            feats = extract_features_separate(img)
            parts = [feats[f] for f in feature_list if f in feats]
            return np.concatenate(parts)
        return _extract

    extract_fn = make_feature_fn(features)

    # 运行数据处理对比
    images_sub, labels_sub = _subsample_balanced(images, labels, min(max_samples, len(labels)), random_seed)

    # 1. 预处理对比
    df_prep = compare_preprocessing_pipelines(
        images, labels, pipelines=selected_pipelines, max_samples=max_samples,
        random_seed=random_seed)

    # 2. 特征对比
    df_feat = compare_feature_subsets(
        images, labels, max_samples=max_samples, random_seed=random_seed)

    # 3. 划分策略对比
    df_split = compare_split_strategies(
        images, labels, max_samples=max_samples, random_seed=random_seed)

    # 生成图表
    # 预处理对比图
    fig_prep, ax_prep = plt.subplots(figsize=(10, 5))
    df_p_sorted = df_prep.sort_values("F1分数")
    ax_prep.barh(range(len(df_p_sorted)), df_p_sorted["F1分数"],
                 color=plt.cm.Blues(np.linspace(0.3, 0.9, len(df_p_sorted))))
    ax_prep.set_yticks(range(len(df_p_sorted)))
    ax_prep.set_yticklabels(df_p_sorted["预处理管线"])
    ax_prep.set_xlabel("F1分数")
    ax_prep.set_title("预处理管线对比")
    plt.tight_layout()

    # 特征对比图
    fig_feat, ax_feat = plt.subplots(figsize=(10, 5))
    df_f_sorted = df_feat.sort_values("F1分数")
    ax_feat.barh(range(len(df_f_sorted)), df_f_sorted["F1分数"],
                 color=plt.cm.Greens(np.linspace(0.3, 0.9, len(df_f_sorted))))
    ax_feat.set_yticks(range(len(df_f_sorted)))
    ax_feat.set_yticklabels(df_f_sorted["特征组合"])
    ax_feat.set_xlabel("F1分数")
    ax_feat.set_title("特征子集对比")
    plt.tight_layout()

    # 划分策略对比图
    fig_split, ax_split = plt.subplots(figsize=(10, 5))
    colors = ['#3498db' if c == '留出法' else '#e74c3c' for c in df_split["类别"]]
    ax_split.bar(range(len(df_split)), df_split["F1分数"], color=colors)
    ax_split.set_xticks(range(len(df_split)))
    ax_split.set_xticklabels(df_split["策略"], rotation=30, ha='right')
    ax_split.set_ylabel("F1分数")
    ax_split.set_title("数据划分策略对比")
    plt.tight_layout()

    # 最优结果
    best_pipeline = df_prep.sort_values("F1分数", ascending=False).iloc[0]["预处理管线"]
    best_features_name = df_feat.sort_values("F1分数", ascending=False).iloc[0]["特征组合"]
    best_split = df_split.sort_values("F1分数", ascending=False).iloc[0]["策略"]

    return {
        "comparison_table": df_prep,  # 主对比表
        "preprocessing_table": df_prep,
        "feature_table": df_feat,
        "split_table": df_split,
        "best_pipeline": best_pipeline,
        "best_features": best_features_name,
        "best_split": best_split,
        "preprocessing_plot": fig_prep,
        "feature_plot": fig_feat,
        "split_plot": fig_split,
    }


print("=" * 70)
print("Gradio 接口：data_processing_handler — 已实现")
print("=" * 70)
print("对应可视化系统 Tab: 数据处理")
print("功能：加载数据 → 应用预处理 → 提取特征 → 运行对比 → 返回结果")
print("返回：dict (含 comparison_table, best_pipeline, plots 等)")
print("=" * 70)